# 02 — Submission (Task 2 Datavidia 11.0, Tim Perintis)

**Alur:** citra → preprocessing → DINOv2 → muat `model/` → prediksi → CSV.

Notebook ini **tidak melatih apa pun** (tidak ada `.fit()`). Semua model dimuat dari `model/` hasil `01_model.ipynb`.

**Output:** `submission/submission_final.csv`

## 0. Library

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib, sklearn

## Pengaturan

In [ ]:
# --- Path (relatif terhadap folder notebook; sesuaikan dengan aturan pengumpulan) ---
DATA_DIR       = Path("Dataset/Fenomena")              # berisi <Id>/pre.jpg dan <Id>/post.jpg
SAMPLE_SUB     = Path("Dataset/sample_submission.csv")
MODEL_DIR      = Path("model")
SUBMISSION_DIR = Path("submission")
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

# --- MODE_UJI: WAJIB False untuk versi yang dikumpulkan ---
MODE_UJI = True
FITUR_TERSIMPAN = Path("../hafizh/v2")     # hanya dipakai saat MODE_UJI
if MODE_UJI:
    SAMPLE_SUB = next(Path(p) for p in ["../../Task-2/sample_submission.csv", "../../sample_submission.csv",
                                        "../Dataset/sample_submission.csv", str(SAMPLE_SUB)] if Path(p).exists())

m = joblib.load(MODEL_DIR / "model.joblib")
print("MODE_UJI =", MODE_UJI, "| blok fitur:", m["fitur_blok"], "| kelas:", m["kelas"])
if m["versi_lib"]["scikit-learn"] != sklearn.__version__:
    print(f"PERINGATAN: model dibuat dengan scikit-learn {m['versi_lib']['scikit-learn']}, sekarang {sklearn.__version__}")

## 1. [A — Hafizh] Preprocessing & ekstraksi fitur (DIKUNCI)

Sel berikut diisi **salinan persis** fungsi preprocessing + ekstraksi yang dikunci Hafizh (sama di `01_model` dan `02_submission`).

Di notebook ini hanya `emb` yang dipakai. Kontrak keluaran `ekstrak_semua(data_dir, ids)`:

| Kunci | Isi |
|---|---|
| `emb` | dict `{"color_pre", "color_post", ...}` → `np.ndarray (N, 1024) float32`, urutan = `ids` |
| `art` | DataFrame `artifact_and_phys_v3` (kolom `Id`, urutan = `ids`) |
| `patch` | DataFrame `patch_change_stats_v3` (kolom `Id`, urutan = `ids`) |
| `phys` | DataFrame `phys_v4` (kolom `Id`, urutan = `ids`) |

Selama `MODE_UJI = True`, fitur dimuat dari file tersimpan A supaya sisa notebook bisa diuji sebelum fungsi A siap.

In [ ]:
# ===== TEMPEL DI SINI: fungsi preprocessing + ekstraksi yang dikunci Hafizh =====
# def ekstrak_semua(data_dir, ids):
#     ...
#     return {"emb": emb, "art": art, "patch": patch, "phys": phys}
# ================================================================================

In [ ]:
sub = pd.read_csv(SAMPLE_SUB)
ids = sub["Id"].to_numpy()
N = len(ids)
assert sub["Id"].is_unique and sub.columns[1] == "Phenomenon"

if MODE_UJI:
    print("PERINGATAN MODE_UJI: fitur dimuat dari file tersimpan, BUKAN diekstrak dari citra")
    fitur = {
        "emb": {b: np.load(FITUR_TERSIMPAN / f"dino_{b}.npy") for b in m["fitur_blok"]},
    }
else:
    assert "ekstrak_semua" in globals(), "fungsi ekstrak_semua dari Hafizh belum ditempel di sel atas"
    fitur = ekstrak_semua(DATA_DIR, ids)

emb = fitur["emb"]
for b in m["fitur_blok"]:
    assert emb[b].shape == (N, 1024) and not np.isnan(emb[b]).any(), f"embedding {b} bermasalah"

print("N =", N, "| blok embedding:", sorted(emb))

## 2. Prediksi dengan model tersimpan

In [ ]:
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)

def bangun_fitur(emb, blok):
    # fitur per foto: L2 per blok embedding, digabung (tanpa event_id)
    return np.hstack([l2(emb[b]) for b in blok])

pred = m["pipeline"].predict(bangun_fitur(emb, m["fitur_blok"]))

SUBMISSION_DIR.mkdir(exist_ok=True)
submission = sub[["Id"]].copy()
submission["Phenomenon"] = pred
submission.to_csv(SUBMISSION_DIR / "submission_final.csv", index=False)
submission["Phenomenon"].value_counts().reindex(KELAS).fillna(0).astype(int).to_frame("n_prediksi")

## 3. Cek akhir

In [ ]:
cek = pd.read_csv(SUBMISSION_DIR / "submission_final.csv")
assert list(cek.columns) == list(sub.columns) and (cek["Id"] == sub["Id"]).all(), "urutan/kolom beda dengan sample_submission"
assert cek["Phenomenon"].notna().all() and set(cek["Phenomenon"]) <= set(KELAS), "ada kelas di luar 6 kelas resmi"

# Prediksi harus sama 100% dengan prediksi saat training (model/pred_train.csv)
ref = pd.read_csv(MODEL_DIR / "pred_train.csv")
assert (ref["Id"] == cek["Id"]).all()
sama = (ref["Phenomenon"] == cek["Phenomenon"]).mean()
print(f"Sama dengan prediksi training: {sama * 100:.2f}%")
assert sama == 1.0, "prediksi 02 beda dengan 01: cek determinisme preprocessing/ekstraksi"
print("Cek akhir LULUS")
if MODE_UJI:
    print("INGAT: ini masih MODE_UJI. Cek 100% yang sebenarnya baru berlaku setelah fungsi A ditempel dan MODE_UJI=False.")